# Contraction analysis of Engineered Heart Tissue (EHT) by pillar displacement tracking

This notebook tracks the displacement of a pillar (a cantilever beam) to which EHT is attached, and analyzes the contraction waveform.

## What it does
1. On the first frame of each video, you select a region of interest (ROI) on the pillar. The ROI is then tracked through all frames with OpenCV's MIL tracker.
2. The x coordinate of the ROI is converted into a time series of pillar displacement (µm).
3. The upper peaks (maximum contraction) and the rising points (where each contraction starts) of the waveform are detected, and the following values are calculated for each beat:
   - **Displacement d**: displacement from the rising point to the upper peak (µm)
   - **Contraction time CT**: time from the point where the displacement reaches 20% of the amplitude to the upper peak (s)
   - **Relaxation time RT**: time from the upper peak to the point where the displacement returns to 20% of the amplitude (s)
4. The contraction force (mN) is calculated from the mean displacement using the cantilever beam equation.
5. The beating rate (Hz) is calculated from the mean interval between peaks.

## Requirements
- Python 3
- `opencv-contrib-python` (required for `cv2.TrackerMIL_create`)
- `numpy`, `scipy`, `matplotlib`, `pandas`

ROI selection uses an OpenCV window, so run this notebook in a local environment that can display a GUI.

## Usage
1. Put the `.avi` videos to be analyzed in `INPUT_DIR`.
2. Edit the values in the "Analysis settings" cell to match your experimental conditions.
3. Run all cells from top to bottom.
4. An ROI selection window opens for each video. Drag a box around the region on the pillar, then press SPACE or ENTER (press c to cancel).
5. While tracking, press Esc to stop early.

## Output
For each video, the folder `OUTPUT_DIR/<video name>/<Left or Right>/` is created with the following files:

| File | Contents |
|---|---|
| `<video name>_<side>_ROI_initial.png` | Initial position of the selected ROI |
| `<video name>_<side>_movie.mp4` | Video with the tracking result overlaid |
| `<video name>_<side>_waveform.csv` | Time (s) and smoothed displacement (µm) for every frame |
| `<video name>_<side>_wave.png` | Displacement waveform with the analysis points (peaks, 20% contraction points, 20% relaxation points) |
| `<video name>_<side>.csv` | d / CT / RT for each beat, their means, contraction force and beating rate |

If an output folder with the same name already exists, the notebook stops with an error to avoid overwriting it. To re-analyze a video, move or delete the old output folder first.

## Import libraries

In [ ]:
import glob
import os
import statistics
import time
import cv2
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal

plt.rcParams["font.size"] = 20  # font size of figures

## Analysis settings
Edit the values in this cell to match your experimental conditions.

In [ ]:
# --- Input / output ---
INPUT_DIR = os.getcwd()                          # folder containing the .avi videos to analyze
OUTPUT_DIR = os.path.join(INPUT_DIR, "output")   # folder where results are saved

# --- Pillar to measure ---
# 'Right' or 'Left'. For 'Left', each frame is rotated by 180 degrees so that it faces the same way as the right pillar
PILLAR_SIDE = "Right"

# --- Image scale (SCALE_PIXEL pixels = SCALE_UM µm) ---
SCALE_PIXEL = 88
SCALE_UM = 1000

# --- Range of beats to analyze ---
# Upper peaks with index PEAK_FROM to PEAK_TO - 1 are analyzed (0-based)
# e.g. PEAK_FROM = 1, PEAK_TO = 4 analyzes 3 beats: the 2nd to the 4th
PEAK_FROM = 1
PEAK_TO = 4

# --- Waveform processing parameters ---
SMOOTHING_WINDOW = 3        # window size of the moving average (frames)
PEAK_PROMINENCE = 5         # prominence used for peak detection (µm)
PEAK_MIN_INTERVAL_S = 0.3   # minimum interval between neighboring peaks (s)

# --- Material property and dimensions of the pillar (cantilever beam) used for the force calculation ---
# Contraction force F = (b * h^3 * E * W) / (4 * l^3)
YOUNG_MODULUS = 2611        # Young's modulus E (MPa = N/mm^2)
PILLAR_WIDTH = 2            # beam width b (mm)
PILLAR_THICKNESS = 0.05     # beam thickness h (mm)
PILLAR_LENGTH = 11.3        # beam length l (mm)

if PILLAR_SIDE not in ("Left", "Right"):
    raise ValueError("PILLAR_SIDE must be 'Left' or 'Right'")

## Function definitions
### 1. ROI tracking

In [ ]:
def get_video_info(cap):
    # Get the frame rate, width and height of the video (the total frame count is only printed for checking)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print("fps:", fps, "/ width:", width, "/ height:", height, "/ total frames:", total)
    return fps, width, height


def roi_corners(roi):
    # From an ROI (x, y, width, height), return the x coordinate of the top-left corner and the top-left / bottom-right corners
    x, y, w, h = map(int, roi)
    return x, (x, y), (x + w, y + h)


def read_frame(cap, side):
    # Read one frame. For the left pillar, rotate it by 180 degrees
    ret, frame = cap.read()
    if ret and side == "Left":
        frame = cv2.rotate(frame, cv2.ROTATE_180)
    return ret, frame


def draw_elapsed_time(frame, count, fps):
    # Write the elapsed time (s) in the top-left corner of the frame
    cv2.putText(frame, "{:.1f}s".format(round(count / fps, 1)), (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 255), 3)


def track_pillar(video_path, side, out_prefix):
    # Select an ROI on the first frame, then track the x coordinate of its top-left corner through all frames
    # Returns: list of x coordinates (pixels), frame rate
    cap = cv2.VideoCapture(video_path)
    ret, frame = read_frame(cap, side)
    fps, width, height = get_video_info(cap)

    # Select the ROI and save its initial position as an image
    roi = cv2.selectROI(frame, False)
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    x, p1, p2 = roi_corners(roi)
    cv2.rectangle(frame, p1, p2, (0, 255, 0), 3)
    cv2.imwrite(out_prefix + "_ROI_initial.png", frame)

    # Prepare to write a video (mp4) with the tracking result overlaid
    fourcc = cv2.VideoWriter_fourcc("m", "p", "4", "v")
    video = cv2.VideoWriter(out_prefix + "_movie.mp4", fourcc, fps, (width, height), True)

    tracker = cv2.TrackerMIL_create()
    tracker.init(frame, roi)
    count = 0
    draw_elapsed_time(frame, count, fps)
    video.write(frame)
    x_list = [x]

    # Track from the second frame to the last frame
    while True:
        start = time.time()
        ret, frame = read_frame(cap, side)
        if not ret:  # stop after the last frame
            break
        success, roi = tracker.update(frame)
        x, p1, p2 = roi_corners(roi)
        cv2.rectangle(frame, p1, p2, (0, 255, 0), 3)
        count += 1
        draw_elapsed_time(frame, count, fps)
        video.write(frame)
        x_list.append(x)
        time.sleep(time.time() - start)  # wait as long as the processing took, so the display does not run too fast
        cv2.imshow("TrackerMIL", frame)
        if cv2.waitKey(1) & 0xff == 27:  # press Esc to stop early
            break

    cap.release()
    video.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    return x_list, fps

### 2. Displacement waveform and peak detection
The pillar moves toward smaller x coordinates when the tissue contracts, so displacement is measured from the maximum x coordinate (the most relaxed position).

In [ ]:
def compute_displacement(x_list, fps):
    # Convert x coordinates (pixels) into a time series of displacement (µm)
    # Returns: time (s), smoothed displacement (µm), raw displacement (µm)
    baseline = np.array(max(x_list))
    dist = (baseline - np.array(x_list)) * SCALE_UM / SCALE_PIXEL
    sec = np.array(range(len(x_list))) / fps
    kernel = np.ones(SMOOTHING_WINDOW) / SMOOTHING_WINDOW
    smoothed = np.convolve(dist, kernel, mode="same")
    return sec, smoothed, dist


def detect_peaks(sec, smoothed, fps):
    # Detect the rising points (where each contraction starts) and the upper peaks (maximum contraction)
    # The upper peaks are aligned so that the i-th upper peak comes after the i-th rising point

    # A rising point is a frame where the displacement has not increased over the previous 2 frames
    # and then increases for 3 consecutive frames
    rising_idx = []
    for m in range(2, len(smoothed) - 3):
        if (smoothed[m - 2] >= smoothed[m - 1] >= smoothed[m]
                and smoothed[m] < smoothed[m + 1] < smoothed[m + 2] < smoothed[m + 3]):
            rising_idx.append(m)

    peak_idx, _ = signal.find_peaks(smoothed, prominence=PEAK_PROMINENCE,
                                    distance=PEAK_MIN_INTERVAL_S * fps)
    if rising_idx[1] > peak_idx[1]:  # the upper peaks are one ahead of the rising points, so drop the first upper peak
        peak_idx = np.delete(peak_idx, 0)

    # Beating rate (Hz) = 1 / mean interval between the upper peaks in the analysis range
    beat_rate_hz = 1 / statistics.mean(np.diff(sec[peak_idx][PEAK_FROM:PEAK_TO]))
    return peak_idx, rising_idx, beat_rate_hz

### 3. Analysis points for each beat
Beat i is defined as the i-th rising point → the i-th upper peak → the (i+1)-th rising point.
- **20% contraction point**: on the way from the rising point to the upper peak, the time when the displacement exceeds "rising point + amplitude × 0.2"
- **20% relaxation point**: on the way from the upper peak to the next rising point, the time when the displacement falls below "next rising point + amplitude × 0.2"

Both times are calculated by linear interpolation between the neighboring frames.

In [ ]:
def round3(value):
    # Round to 3 decimal places (rounds correctly in decimal, even for numpy values)
    return float("{:.3f}".format(value))


def analyze_beats(sec, smoothed, peak_idx, rising_idx):
    # For each beat, find the time and displacement of the upper peak, the 20% contraction point and the 20% relaxation point
    # Returns: {beat index: dict of analysis points} (values rounded to 3 decimal places)
    beats = {}
    for i in range(PEAK_FROM, PEAK_TO):
        peak_d = smoothed[peak_idx[i]]
        rising_d = smoothed[rising_idx[i]]           # rising point at the start of this beat
        next_rising_d = smoothed[rising_idx[i + 1]]  # rising point at the start of the next beat
        rise = peak_d - rising_d                     # amplitude on the contraction side
        fall = peak_d - next_rising_d                # amplitude on the relaxation side

        # 20% contraction point: the first frame between the rising point and the upper peak that exceeds the threshold
        cont_level = rise * 0.2 + rising_d
        rise_frames = np.arange(rising_idx[i], peak_idx[i])
        c = min(rise_frames[smoothed[rise_frames] - cont_level > 0])
        cont_t = (cont_level - smoothed[c - 1]) * (sec[c] - sec[c - 1]) / (smoothed[c] - smoothed[c - 1]) + sec[c - 1]

        # 20% relaxation point: the first frame between the upper peak and the next rising point that falls below the threshold
        relax_level = fall * 0.2 + next_rising_d
        fall_frames = np.arange(peak_idx[i], rising_idx[i + 1])
        r = min(fall_frames[smoothed[fall_frames] - relax_level < 0])
        relax_t = (relax_level - smoothed[r]) * (sec[r - 1] - sec[r]) / (smoothed[r - 1] - smoothed[r]) + sec[r]

        beats[i] = {
            "peak_t": round3(sec[peak_idx[i]]),  # time of the upper peak (s)
            "amplitude": round3(rise),           # displacement d (µm)
            "cont_t": round3(cont_t),            # time of the 20% contraction point (s)
            "cont_d": round3(cont_level),        # displacement at the 20% contraction point (µm)
            "relax_t": round3(relax_t),          # time of the 20% relaxation point (s)
            "relax_d": round3(relax_level),      # displacement at the 20% relaxation point (µm)
        }
    return beats

### 4. Plotting the waveform and summarizing the results

In [ ]:
def plot_wave(sec, smoothed, dist, peak_idx, rising_idx, beats, save_path):
    # Plot the smoothed displacement waveform with the detected peaks and analysis points, and save it
    # Red circles: upper peaks and analysis points, blue circles: rising points
    fig = plt.figure(figsize=(14.0, 6.0))
    ax = fig.add_subplot(111)
    ax.set_ylim(min(dist) - max(dist) * 0.1, max(dist) + max(dist) * 0.2)
    ax.plot(sec, smoothed, color="k")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("distance (um)")
    ax.scatter(sec[peak_idx], smoothed[peak_idx], facecolor="None", edgecolors="red")
    ax.scatter(sec[rising_idx], smoothed[rising_idx], facecolor="None", edgecolors="blue")

    for i, b in beats.items():
        rising_d = smoothed[rising_idx[i]]
        # Amplitude (rising point → upper peak)
        ax.scatter(b["peak_t"], b["amplitude"] + rising_d, facecolor="None", edgecolors="red")
        ax.vlines(b["peak_t"], rising_d, b["amplitude"] + rising_d, "r")
        ax.hlines(rising_d, sec[rising_idx[i]], b["peak_t"], linestyle="dashed", colors="r")
        # Contraction time CT (20% contraction point → upper peak)
        ax.scatter(b["cont_t"], b["cont_d"], facecolor="None", edgecolors="red")
        ax.hlines(b["cont_d"], b["cont_t"], b["peak_t"], colors="r")
        # Relaxation time RT (upper peak → 20% relaxation point)
        ax.scatter(b["relax_t"], b["relax_d"], facecolor="None", edgecolors="red")
        ax.hlines(b["relax_d"], b["peak_t"], b["relax_t"], colors="r")

    fig.savefig(save_path)
    plt.show()


def summarize(beats, beat_rate_hz, video_path, side):
    # Put d / CT / RT for each beat, their means, the contraction force and the beating rate into a one-row table
    row = {"LR": side, "filename": video_path}
    for i, b in beats.items():
        row[f"peak{i}_d"] = b["amplitude"]
    for i, b in beats.items():
        row[f"peak{i}_CT"] = round3(b["peak_t"] - b["cont_t"])
    for i, b in beats.items():
        row[f"peak{i}_RT"] = round3(b["relax_t"] - b["peak_t"])

    mean_d = sum(b["amplitude"] for b in beats.values()) / len(beats)
    mean_ct = sum(row[f"peak{i}_CT"] for i in beats) / len(beats)
    mean_rt = sum(row[f"peak{i}_RT"] for i in beats) / len(beats)
    row["mean_d"] = round3(mean_d)
    row["mean_CT"] = round3(mean_ct)
    row["mean_RT"] = round3(mean_rt)

    # Contraction force F = (b * h^3 * E * W) / (4 * l^3)
    # where b is the pillar width (2 mm), h is the pillar thickness (0.05 mm), E is the Young's modulus (2611 MPa), W is the pillar displacement (mm), and l is the pillar length (11.3 mm)
    # W is converted from µm to mm (/1000), and F from N to mN (*1000)
    row["Force (mN)"] = (PILLAR_WIDTH * (PILLAR_THICKNESS ** 3) * YOUNG_MODULUS * mean_d
                             / 4 / PILLAR_LENGTH ** 3 * 1000 / 1000)
    row["Hz"] = beat_rate_hz
    return pd.DataFrame([row])

## Run the analysis
All `.avi` videos in `INPUT_DIR` are analyzed in file name order.

In [ ]:
video_paths = sorted(glob.glob(os.path.join(INPUT_DIR, "*.avi")))
print("Total Count of Files:", len(video_paths))
for n, path in enumerate(video_paths):
    print(n, os.path.basename(path))

In [ ]:
for video_path in video_paths:
    print(video_path)
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    out_dir = os.path.join(OUTPUT_DIR, video_name, PILLAR_SIDE)
    os.makedirs(out_dir, exist_ok=False)  # raise an error if the folder exists, so existing results are not overwritten
    out_prefix = os.path.join(out_dir, f"{video_name}_{PILLAR_SIDE}")

    x_list, fps = track_pillar(video_path, PILLAR_SIDE, out_prefix)
    sec, smoothed, dist = compute_displacement(x_list, fps)
    waveform = pd.DataFrame({"time (s)": sec, "smoothed_distance (um)": smoothed})
    waveform.to_csv(out_prefix + "_waveform.csv", index=False)
    peak_idx, rising_idx, beat_rate_hz = detect_peaks(sec, smoothed, fps)
    beats = analyze_beats(sec, smoothed, peak_idx, rising_idx)
    plot_wave(sec, smoothed, dist, peak_idx, rising_idx, beats, out_prefix + "_wave.png")

    result = summarize(beats, beat_rate_hz, video_path, PILLAR_SIDE)
    print(result)
    result.to_csv(out_prefix + ".csv", index=False)